In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def ask(prompt, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

## 1. The Classic Example

Famous problem that tricks many models without CoT.

In [ ]:
# ❌ Without CoT - Often gets it wrong!
prompt_no_cot = """
A bat and a ball cost $1.10 in total.
The bat costs $1.00 more than the ball.
How much does the ball cost?
"""

print("Without CoT:")
print(ask(prompt_no_cot))
print()

# ✅ With CoT - Much better!
prompt_with_cot = prompt_no_cot + "\nLet's think step by step:"

print("With CoT:")
print(ask(prompt_with_cot))

## 2. Zero-Shot CoT

Just add "Let's think step by step" - that's it!

In [ ]:
# Math problem
problem = """
If a train travels 120 miles in 2 hours, and then 90 miles in 1.5 hours,
what is its average speed for the entire journey?

Let's think step by step:
"""

print(ask(problem))

In [ ]:
# Logic puzzle
puzzle = """
All roses are flowers.
Some flowers fade quickly.
Therefore, do all roses fade quickly?

Let's reason through this step by step:
"""

print(ask(puzzle))

## 3. Few-Shot CoT

Show examples of step-by-step reasoning.

In [ ]:
prompt = """
Solve these word problems step-by-step:

Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
   Each can has 3 tennis balls. How many tennis balls does he have now?
A: Roger started with 5 balls.
   Step 1: He bought 2 cans, each with 3 balls: 2 × 3 = 6 balls
   Step 2: Add to his original: 5 + 6 = 11 balls
   Answer: 11 tennis balls

Q: The cafeteria had 23 apples. If they used 20 to make lunch and
   bought 6 more, how many apples do they have?
A: Started with 23 apples.
   Step 1: Used 20 for lunch: 23 - 20 = 3 apples left
   Step 2: Bought 6 more: 3 + 6 = 9 apples
   Answer: 9 apples

Q: A parking lot has 12 spaces. 8 are occupied. 3 cars leave and
   5 new cars arrive. How many spaces are now occupied?
A:
"""

print(ask(prompt))

## 4. Self-Consistency

Generate multiple reasoning paths, take majority vote.

In [ ]:
from collections import Counter
import re

def extract_answer(text):
    """Extract final numerical answer."""
    # Look for "Answer: X" pattern
    match = re.search(r'Answer:?\s*([\d.]+)', text, re.IGNORECASE)
    if match:
        return match.group(1)
    # Look for last number
    numbers = re.findall(r'\b\d+\.?\d*\b', text)
    return numbers[-1] if numbers else None

problem = """
A farmer has 15 chickens. Each chicken lays 2 eggs per day.
The farmer sells eggs in cartons of 6.
How many full cartons can he fill in one day?

Let's solve this step by step:
"""

# Generate 5 different reasoning paths
answers = []
print("Generating 5 reasoning paths...\n")

for i in range(5):
    response = ask(problem, temperature=0.7)  # Higher temperature for diversity
    answer = extract_answer(response)
    answers.append(answer)
    print(f"Path {i+1}: Answer = {answer}")

# Majority vote
counter = Counter(answers)
final_answer = counter.most_common(1)[0][0]

print(f"\nMajority vote: {final_answer}")
print(f"Vote distribution: {dict(counter)}")

## 5. Structured CoT

Force specific reasoning structure.

In [ ]:
prompt = """
Analyze this product review using the following structure:

Review: "The laptop is fast and the screen is beautiful, but it gets very hot
and the battery only lasts 3 hours. For $1200, I expected better."

Please analyze:

Step 1 - Identify positive aspects:
Step 2 - Identify negative aspects:
Step 3 - Consider price-value relationship:
Step 4 - Overall sentiment (positive/negative/mixed):
Step 5 - Recommendation (buy/don't buy/consider alternatives):
"""

print(ask(prompt))

## 6. CoT for Code Debugging

Make models reason about bugs.

In [ ]:
code_debug = """
This function is supposed to find the maximum value in a list, but it's not working:

```python
def find_max(numbers):
    max_num = 0
    for num in numbers:
        if num > max_num:
            max_num = num
    return max_num
```

Test case that fails: find_max([-5, -2, -8]) returns 0, but should return -2

Let's debug this step by step:
1. What is the function trying to do?
2. What does it do on the failing test case?
3. Why does it fail?
4. How to fix it?
"""

print(ask(code_debug))

## 7. Least-to-Most Prompting

Break complex problems into subproblems.

In [ ]:
# First: Decompose the problem
decompose = """
Task: Plan a 3-day trip to Paris for a family of 4 on a $3000 budget.

First, let's break this into smaller questions we need to answer:
"""

print("=== Step 1: Decomposition ===")
subproblems = ask(decompose)
print(subproblems)

# Second: Solve each subproblem
solve_first = f"""
Based on these subproblems:
{subproblems}

Let's solve the first one in detail:
"""

print("\n=== Step 2: Solving First Subproblem ===")
print(ask(solve_first))

## Best Practices

### When to Use CoT

✅ **DO use CoT for:**
- Math and arithmetic
- Logic puzzles
- Commonsense reasoning
- Multi-step tasks
- Complex analysis

❌ **DON'T use CoT for:**
- Simple lookups
- Straightforward classification
- When speed > accuracy
- Very clear, simple tasks

### Tips

1. **"Let's think step by step"** works surprisingly well
2. **Temperature = 0** for consistent reasoning
3. **Self-consistency** for important decisions (5-10 samples)
4. **Structure** your reasoning format for complex tasks
5. **Few-shot examples** improve quality significantly

### Cost vs. Quality

CoT uses more tokens (2-5x), but:
- Higher accuracy (often 20-30% improvement)
- Easier debugging
- Better for high-stakes decisions
- Worth it for complex reasoning

## Exercise: Build a CoT Solver

Create a function that solves math word problems with CoT.

In [ ]:
def solve_math_problem(problem, use_self_consistency=False, n_samples=5):
    """Solve math problem with CoT.
    
    Args:
        problem: Math word problem as string
        use_self_consistency: Whether to use multiple samples
        n_samples: Number of samples for self-consistency
    
    Returns:
        Final answer
    """
    prompt = f"{problem}\n\nLet's solve this step by step:"
    
    if use_self_consistency:
        answers = []
        for _ in range(n_samples):
            response = ask(prompt, temperature=0.7)
            answer = extract_answer(response)
            answers.append(answer)
        
        # Return majority vote
        counter = Counter(answers)
        return counter.most_common(1)[0][0]
    else:
        response = ask(prompt)
        return extract_answer(response)

# Test it
problem = """
A store sells notebooks for $3 each and pens for $2 each.
Sarah bought 5 notebooks and 8 pens.
She paid with a $50 bill.
How much change should she receive?
"""

print("Simple CoT:")
print(solve_math_problem(problem))

print("\nWith self-consistency:")
print(solve_math_problem(problem, use_self_consistency=True))

## Key Takeaways

1. **CoT = Better Reasoning**: Step-by-step improves accuracy
2. **"Let's think step by step"**: Simple magic phrase
3. **Self-consistency**: Multiple paths → robust answers
4. **Structure helps**: Guide the reasoning format
5. **Trade-off**: More tokens but better results

## Next Steps

- `03_react_prompting.ipynb` - Add tool use to reasoning
- `04_tree_of_thoughts.ipynb` - Explore multiple reasoning branches
- `06_optimization.ipynb` - Test and improve your prompts